## **1. Environment Setup and Data Restoration**
Detecting GPU capabilities, setting global hyperparameters, and reconstructing the data pipeline.

### **1.1: Imports, relative paths, and hardware/GPU VRAM detection**

In [1]:
import os
import pickle
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Set up dynamic relative paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATASET_PATH = os.path.join(BASE_DIR, "Dataset")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "models")

# Ensure the models directory exists for saving weights later
os.makedirs(MODELS_DIR, exist_ok=True)

# Hardware Detection and VRAM Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Hardware Configuration:")
print(f"Active Device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name}")
    print(f"Total VRAM: {vram_gb:.2f} GB")
else:
    print("Warning: Running on CPU. Training will be extremely slow.")

Hardware Configuration:
Active Device: cuda
GPU Model: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM: 6.00 GB


### **Cell 1.2: Dedicated Hyperparameters `Batch Size, Epochs, and Learning Rates`**


In [2]:
# Load dataset metadata saved during preprocessing to retrieve base sizes
info_path = os.path.join(RESULTS_DIR, 'dataset_info.pkl')
with open(info_path, 'rb') as f:
    dataset_info = pickle.load(f)

# Global Training Hyperparameters
# Lower BATCH_SIZE to 32 or 16 if you encounter CUDA Out of Memory errors
BATCH_SIZE = dataset_info['batch_size'] 
INPUT_SIZE = dataset_info['input_size']
BASE_LEARNING_RATE = 0.001

# Adjust Epochs as per GPU Power
EPOCHS_STANDARD = 10

print("Hyperparameter Configuration:")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Input Size: {INPUT_SIZE}x{INPUT_SIZE}")
print(f"Base Learning Rate: {BASE_LEARNING_RATE}")
print(f"Total Epochs: {EPOCHS_STANDARD}")

Hyperparameter Configuration:
Batch Size: 64
Input Size: 224x224
Base Learning Rate: 0.001
Total Epochs: 10


### **Cell 1.3: Data restoration (loading `dataset_info.pkl` and initializing DataLoaders)**

In [3]:
# Reconstruct standard transformation pipelines
train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Reconstruct the dataset class
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_label = {'fake': 0, 'real': 1}
        
        for class_name, label in self.class_to_label.items():
            class_path = os.path.join(root_dir, class_name)
            if os.path.exists(class_path):
                for img_file in os.listdir(class_path):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        self.images.append(os.path.join(class_path, img_file))
                        self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            return self.__getitem__((idx + 1) % len(self.images))
            
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Initialize Datasets and DataLoaders
train_dataset = DeepfakeDataset(os.path.join(DATASET_PATH, 'train'), transform=train_transform)
val_dataset = DeepfakeDataset(os.path.join(DATASET_PATH, 'validation'), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print("Data Restoration Complete:")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Data Restoration Complete:
Train batches: 2188
Validation batches: 617


## **2. Model Architecture**
Defining the Custom CNN architecture from scratch and verifying its parameter count.

In [4]:
import torch.nn as nn

# Define the custom Convolutional Neural Network architecture
class CustomCNN(nn.Module):
    def __init__(self, num_classes=1):
        super(CustomCNN, self).__init__()
        
        # Convolutional feature extractor
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(0.25)
        )
        
        # Flattened size based on 224x224 input passing through 4 poolings (224 / 16 = 14)
        self.feature_size = 256 * 14 * 14
        
        # Fully connected classification head
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_size, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Initialize the model
custom_cnn = CustomCNN(num_classes=1)

# Output the architecture parameter summary
print("Architecture Initialized:")
print(f"Custom CNN Total Params: {sum(p.numel() for p in custom_cnn.parameters()):,}")

Architecture Initialized:
Custom CNN Total Params: 26,997,921


## **3. Training Engine**
Implementing a clean `Trainer` class tailored for our CNN, handling the training loop, automatic checkpointing, and learning rate scheduling.

In [5]:
import os
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm

# Trainer class strictly configured for the Custom CNN architecture
class Trainer:
    def __init__(self, model, model_name, device, learning_rate=0.001):
        self.model = model.to(device)
        self.model_name = model_name
        self.device = device
        
        # BCEWithLogitsLoss provides numeric stability for binary classification
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        # Reduce learning rate when validation loss stops improving
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=3, factor=0.5
        )
        
        self.history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        self.best_val_acc = 0
        self.start_epoch = 0

    # Save full training state for potential resuming
    def save_checkpoint(self, filepath, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_val_acc': self.best_val_acc,
            'history': self.history
        }
        torch.save(checkpoint, filepath)

    # Load previous training state
    def load_checkpoint(self, filepath):
        checkpoint = torch.load(filepath, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        self.start_epoch = checkpoint['epoch'] + 1
        self.best_val_acc = checkpoint['best_val_acc']
        self.history = checkpoint['history']

    # Execute a single training epoch
    def train_epoch(self, loader):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in tqdm(loader, desc=f"Training {self.model_name}"):
            images, labels = images.to(self.device), labels.to(self.device)
            labels = labels.float().unsqueeze(1)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
        return running_loss / len(loader), 100. * correct / total

    # Execute a single validation epoch
    def validate_epoch(self, loader):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in tqdm(loader, desc=f"Validating {self.model_name}"):
                images, labels = images.to(self.device), labels.to(self.device)
                labels = labels.float().unsqueeze(1)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                predicted = (torch.sigmoid(outputs) > 0.5).float()
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
                
        return running_loss / len(loader), 100. * correct / total

    # Main training loop execution
    def train(self, train_loader, val_loader, epochs, save_path=None, resume_path=None, save_every=1):
        if resume_path and os.path.exists(resume_path):
            self.load_checkpoint(resume_path)
            print(f"Resuming {self.model_name} from epoch {self.start_epoch}")
            
        for epoch in range(self.start_epoch, epochs):
            print(f"\nEpoch {epoch+1}/{epochs}:")
            
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate_epoch(val_loader)
            
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            
            self.scheduler.step(val_loss)
            current_lr = self.optimizer.param_groups[0]['lr']
            
            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
            print(f"Learning Rate: {current_lr:.6f}")
            
            if val_acc > self.best_val_acc and save_path:
                self.best_val_acc = val_acc
                # Save the best model state with history for full restoration
                torch.save({
                    'model_state_dict': self.model.state_dict(),
                    'val_acc': val_acc,
                    'epoch': epoch,
                    'history': self.history
                }, save_path)
                print(f"Saved new best model with accuracy: {val_acc:.2f}%")
                
            if resume_path and (epoch + 1) % save_every == 0:
                self.save_checkpoint(resume_path, epoch)
                
        print(f"\nTraining Complete. Best Validation Accuracy: {self.best_val_acc:.2f}%")
        return self.history

print("Trainer Class Initialized.")

Trainer Class Initialized.


## **4. Execution**
Initializing the trainer and starting the Custom CNN training loop.

In [ ]:
# Define relative save paths in the models directory
custom_cnn_best_path = os.path.join(MODELS_DIR, 'custom_cnn_best.pth')
custom_cnn_checkpoint_path = os.path.join(MODELS_DIR, 'custom_cnn_checkpoint.pth')

# Initialize trainer using the global BASE_LEARNING_RATE
cnn_trainer = Trainer(
    model=custom_cnn, 
    model_name="CustomCNN", 
    device=device, 
    learning_rate=BASE_LEARNING_RATE
)

print("Starting Custom CNN Training Pipeline.")

# Execute training loop using the global EPOCHS_STANDARD
cnn_history = cnn_trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS_STANDARD,
    save_path=custom_cnn_best_path,
    resume_path=custom_cnn_checkpoint_path,
    save_every=1
)

Starting Custom CNN Training Pipeline.

Epoch 1/10:


Validating CustomCNN: 100%|██████████████████████████████████████████████████████████| 617/617 [11:22<00:00,  1.11s/it]


Train Loss: 0.3209 | Train Acc: 84.92%
Val Loss: 0.1910 | Val Acc: 92.00%
Learning Rate: 0.001000
Saved new best model with accuracy: 92.00%

Epoch 2/10:


Validating CustomCNN: 100%|██████████████████████████████████████████████████████████| 617/617 [02:16<00:00,  4.51it/s]


Train Loss: 0.1741 | Train Acc: 92.83%
Val Loss: 0.1575 | Val Acc: 93.49%
Learning Rate: 0.001000
Saved new best model with accuracy: 93.49%

Epoch 3/10:


...

Validating CustomCNN: 100%|██████████████████████████████████████████████████████████| 617/617 [02:20<00:00,  4.39it/s]


Train Loss: 0.1133 | Train Acc: 95.55%
Val Loss: 0.1271 | Val Acc: 94.91%
Learning Rate: 0.001000

Epoch 7/10:


Validating CustomCNN: 100%|██████████████████████████████████████████████████████████| 617/617 [02:09<00:00,  4.77it/s]


Train Loss: 0.1119 | Train Acc: 95.61%
Val Loss: 0.1060 | Val Acc: 95.76%
Learning Rate: 0.001000
Saved new best model with accuracy: 95.76%

Epoch 8/10:


Training CustomCNN:   0%|▏                                                            | 7/2188 [00:04<24:01,  1.51it/s]

 ### **Cell 4.1: Load Weights and Convert to TorchScript**

In [7]:
best_weights_path = os.path.join(MODELS_DIR, 'custom_cnn_best.pth')

# Load the checkpoint from best epoch
checkpoint = torch.load(best_weights_path, map_location=device, weights_only=True)

# Apply learned weights to the architecture
custom_cnn.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded weights from Epoch {checkpoint['epoch'] + 1} with Validation Acc: {checkpoint['val_acc']:.2f}%")

# Move model to correct device and set evaluation mode
custom_cnn = custom_cnn.to(device)
custom_cnn.eval()

# Create matching dummy input and trace the model
dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE).to(device)
scripted_model = torch.jit.trace(custom_cnn, dummy_input)

# Save the standalone model
standalone_path = os.path.join(MODELS_DIR, 'custom_cnn_standalone.pt')
scripted_model.save(standalone_path)
print(f"Standalone TorchScript model saved successfully at: {standalone_path}")

Loaded weights from Epoch 7 with Validation Acc: 95.76%
Standalone TorchScript model saved successfully at: C:\Users\javed\Home\Deepfake-Detection\models\custom_cnn_standalone.pt
